# Ablation Study — Model Definitions

Notebook này định nghĩa sáu mô hình được sử dụng trong thực nghiệm Ablation Study của DeepKoopFormer:

- DLinear
- LSTM
- PatchTST
- PatchTST + Linear Latent
- KoopPatchTST without Lyapunov
- KoopPatchTST with Lyapunov

Các mô hình sử dụng cùng cấu hình thực nghiệm để phục vụ việc so sánh. Notebook chỉ chứa định nghĩa kiến trúc, không thực hiện huấn luyện.

## 1. Environment and Model Configuration

Thiết lập các thư viện PyTorch, seed ngẫu nhiên và siêu tham số cần thiết để khởi tạo mô hình.

Các tập dữ liệu đã được chuẩn bị trong `01_ablation_data.ipynb`. Phần này không tái tạo dữ liệu và không thay đổi kết quả thực nghiệm.

In [1]:
from pathlib import Path
import random
import math

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

# 1. Reproducibility
SEED = 7

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# 2. Ablation grid
PATCH_LENGTHS = [70, 80, 90, 100, 110, 120, 130]
HORIZONS = [2, 4, 6, 8, 10, 12, 14, 16]

# 3. Model hyperparameters
D_MODEL = 96
D_LATENT = 96
NUM_ENCODER_LAYERS = 3
NUM_ATTENTION_HEADS = 4
DIM_FEEDFORWARD = 96
LSTM_HIDDEN_DIM = 96
LSTM_NUM_LAYERS = 2

RHO_MAX = 0.99
LYAP_WEIGHT = 0.1

EPOCHS = 4000
LEARNING_RATE = 3e-4

# 4. Six ablation variants
MODEL_VARIANTS = [
    "DLinear",
    "LSTM",
    "PatchTST",
    "PatchTST+LinearLatent",
    "KoopPatchTST_NoLyap",
    "KoopPatchTST_Full",
]

## 2. Koopman and Linear Latent Operators

Phần này định nghĩa hai phép biến đổi trong không gian tiềm ẩn (latent space).

**StrictStableKoopmanOperator:** Sử dụng phân rã Orthogonal-Diagonal-Orthogonal (ODO) để xây dựng toán tử Koopman với các giá trị kỳ dị bị chặn bởi `rho_max`. Cách tham số hóa này đảm bảo toán tử có tính ổn định theo ràng buộc đã thiết lập.

**LinearLatentPropagator:** Sử dụng phép biến đổi tuyến tính không ràng buộc để cập nhật latent representation. Đây là thành phần được sử dụng trong biến thể PatchTST+LinearLatent.

Hai toán tử cho phép so sánh ảnh hưởng của ràng buộc ổn định Koopman trong thực nghiệm Ablation.

In [2]:
# 1. Orthogonalization
def _orth(w: torch.Tensor) -> torch.Tensor:
    return torch.linalg.qr(w)[0]


# 2. Strictly Stable Koopman Operator
class StrictStableKoopmanOperator(nn.Module):
    """
    Orthogonal-Diagonal-Orthogonal (ODO)
    parameterization with singular values < rho_max.
    """

    def __init__(
        self,
        latent_dim: int,
        ρ_max: float = 0.99
    ):
        super().__init__()

        self.U_raw = nn.Parameter(
            torch.randn(latent_dim, latent_dim)
        )

        self.V_raw = nn.Parameter(
            torch.randn(latent_dim, latent_dim)
        )

        self.S_raw = nn.Parameter(
            torch.randn(latent_dim)
        )

        self.ρ_max = ρ_max

    def forward(self, z: torch.Tensor):

        U = _orth(self.U_raw)
        V = _orth(self.V_raw)

        Σ = torch.sigmoid(self.S_raw) * self.ρ_max

        K = U @ torch.diag(Σ) @ V.T

        z_next = z @ K.T

        return z_next, K, Σ


# 3. Unconstrained Linear Latent Propagator
class LinearLatentPropagator(nn.Module):
    """
    Unconstrained latent transformation:
        z_next = W(z)

    Used by PatchTST+LinearLatent ablation.
    """

    def __init__(self, latent_dim: int):
        super().__init__()

        self.W = nn.Linear(
            latent_dim,
            latent_dim,
            bias=False
        )

    def forward(self, z: torch.Tensor):

        K = self.W.weight.T
        Σ = None

        return self.W(z), K, Σ

## 3. Positional Encoding and Patch Embedding

Phần này định nghĩa hai thành phần dùng để biểu diễn dữ liệu chuỗi thời gian trước khi đưa vào Transformer.

**SinCosPosEnc:** Bổ sung thông tin vị trí vào các patch embedding bằng sinusoidal positional encoding.

**PatchEmbed1D:** Sử dụng phép tích chập một chiều (`Conv1d`) để chuyển chuỗi thời gian đầu vào thành các patch embedding có số chiều `d_model`.

Đầu vào có dạng `(batch, sequence_length, channels)` và đầu ra của Patch Embedding có dạng `(batch, num_patches, d_model)`.

Trong cấu hình Ablation hiện tại, patch length đồng thời được sử dụng làm stride.

In [3]:
# 1. Sinusoidal Positional Encoding
class SinCosPosEnc(nn.Module):

    def __init__(
        self,
        d_model: int,
        max_len: int = 10_000
    ):
        super().__init__()

        pos = torch.arange(
            max_len
        ).float().unsqueeze(1)

        div = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-np.log(10_000.0) / d_model)
        )

        pe = torch.zeros(max_len, d_model)

        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)

        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:x.size(1)]


# 2. One-dimensional Patch Embedding
class PatchEmbed1D(nn.Module):

    def __init__(
        self,
        in_ch: int,
        d_model: int,
        patch_len: int,
        stride: int
    ):
        super().__init__()

        self.conv = nn.Conv1d(
            in_ch,
            d_model,
            kernel_size=patch_len,
            stride=stride
        )

    def forward(self, x):

        # (B, T, C) -> (B, C, T)
        x = x.permute(0, 2, 1)

        # (B, C, T) -> (B, d_model, P)
        x = self.conv(x)

        # (B, d_model, P) -> (B, P, d_model)
        return x.permute(0, 2, 1)

## 4. PatchTST Backbone

Phần này định nghĩa backbone dựa trên Transformer dùng chung cho các biến thể PatchTST trong Ablation Study.

Kiến trúc bao gồm:

- Patch Embedding bằng Conv1d.
- Sinusoidal Positional Encoding.
- Learnable CLS token.
- Transformer Encoder.
- Trích xuất CLS representation làm latent representation.

Cấu hình Ablation sử dụng `d_model=96`, 3 Transformer Encoder layers, 4 attention heads và feedforward dimension bằng 96.

Trong các cấu hình được tạo bởi Model Factory, sequence length bằng patch length nên mỗi chuỗi đầu vào được biểu diễn bằng một patch trước khi thêm CLS token.

Backbone trả về latent representation phục vụ trực tiếp cho forecasting head hoặc phép biến đổi latent.

In [4]:
# 1. PatchTST Backbone
class PatchTST_Backbone(nn.Module):

    def __init__(
        self,
        input_dim: int,
        seq_len: int,
        patch_len: int,
        d_model: int = 64,
        num_layers: int = 3,
        num_heads: int = 4,
        dim_ff: int = 96
    ):
        super().__init__()

        # Patch Embedding
        self.patch = PatchEmbed1D(
            input_dim,
            d_model,
            patch_len,
            patch_len
        )

        # Positional Encoding
        n_patches = int(
            np.ceil(seq_len / patch_len)
        )

        self.pos = SinCosPosEnc(
            d_model,
            max_len=n_patches + 1
        )

        # Transformer Encoder
        enc = nn.TransformerEncoderLayer(
            d_model,
            num_heads,
            dim_ff,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            enc,
            num_layers
        )

        # Learnable CLS token
        self.cls = nn.Parameter(
            torch.randn(1, 1, d_model)
        )

    def forward(self, x):

        # (B, T, C) -> (B, P, d_model)
        x = self.patch(x)

        # Expand CLS token for the batch
        cls = self.cls.expand(
            x.size(0), -1, -1
        )

        # (B, P, d_model) -> (B, 1+P, d_model)
        x = torch.cat(
            [cls, x],
            dim=1
        )

        # Transformer Encoder
        x = self.encoder(
            self.pos(x)
        )

        # Return CLS representation
        return x[:, 0]

## 5. PatchTST Ablation Variants

Phần này định nghĩa các biến thể mô hình dựa trên PatchTST Backbone nhằm đánh giá vai trò của phép biến đổi latent và ràng buộc ổn định.

**PatchTSTBaseline:** Sử dụng trực tiếp latent representation từ Transformer Backbone để dự báo, không có latent propagation operator.

**PatchTST_UnconstrainedLatent:** Bổ sung một phép biến đổi tuyến tính không ràng buộc giữa backbone và forecasting head.

**Koopformer_PatchTST:** Sử dụng StrictStableKoopmanOperator để biến đổi latent representation trước khi dự báo.

Hai biến thể `KoopPatchTST_NoLyap` và `KoopPatchTST_Full` dùng chung kiến trúc Koopformer_PatchTST nhưng khác nhau ở hệ số chính quy hóa Lyapunov được gán trong Model Factory.

Các mô hình hỗ trợ trả về prediction, latent trước biến đổi và latent sau biến đổi thông qua `return_latents=True`.

In [5]:
# 1. Standard PatchTST Baseline
class PatchTSTBaseline(nn.Module):
    """
    PatchTST backbone with direct forecasting head.
    No latent propagation operator.
    """

    def __init__(
        self, input_dim, seq_len, horizon,
        patch_len, d_model=64
    ):
        super().__init__()

        self.backbone = PatchTST_Backbone(
            input_dim,
            seq_len,
            patch_len=patch_len,
            d_model=d_model
        )

        self.fc = nn.Linear(
            d_model,
            horizon * input_dim
        )

    def forward(self, x, return_latents=False):

        z = self.backbone(x)
        pred = self.fc(z)

        if return_latents:
            return pred, z, z

        return pred


# 2. PatchTST + Unconstrained Linear Latent
class PatchTST_UnconstrainedLatent(nn.Module):
    """
    PatchTST with an unconstrained linear latent
    transformation and no Lyapunov regularization.
    """

    def __init__(
        self, input_dim, seq_len, horizon,
        patch_len, d_model=64
    ):
        super().__init__()

        self.backbone = PatchTST_Backbone(
            input_dim,
            seq_len,
            patch_len=patch_len,
            d_model=d_model
        )

        self.koop = LinearLatentPropagator(d_model)

        self.fc = nn.Linear(
            d_model,
            horizon * input_dim
        )

    def forward(self, x, return_latents=False):

        z = self.backbone(x)

        z_next, _, _ = self.koop(z)

        pred = self.fc(z_next)

        if return_latents:
            return pred, z, z_next

        return pred


# 3. PatchTST + Strict Stable Koopman
class Koopformer_PatchTST(nn.Module):
    """
    PatchTST with strictly stable Koopman operator.

    Used for:
        KoopPatchTST_NoLyap (lambda = 0.0)
        KoopPatchTST_Full   (lambda = 0.1)
    """

    def __init__(
        self, input_dim, seq_len, horizon,
        patch_len, d_model=64
    ):
        super().__init__()

        self.backbone = PatchTST_Backbone(
            input_dim,
            seq_len,
            patch_len=patch_len,
            d_model=d_model
        )

        self.koop = StrictStableKoopmanOperator(
            d_model
        )

        self.fc = nn.Linear(
            d_model,
            horizon * input_dim
        )

    def forward(self, x, return_latents=False):

        z = self.backbone(x)

        z_next, _, _ = self.koop(z)

        pred = self.fc(z_next)

        if return_latents:
            return pred, z, z_next

        return pred

## 6. Baseline Models — LSTM and DLinear

Phần này định nghĩa hai mô hình baseline không sử dụng kiến trúc Koopman-Transformer.

**SimpleLSTMForecaster:** Sử dụng mạng LSTM để học đặc trưng chuỗi thời gian, sau đó dùng fully connected layer để tạo dự báo đa bước. Cấu hình thực nghiệm sử dụng hidden dimension bằng 96 và 2 LSTM layers.

**DLinearForecaster:** Sử dụng các phép tích chập một chiều riêng theo từng channel để tạo dự báo. Trong implementation hiện tại, mỗi channel tạo ra một giá trị và mở rộng giá trị đó theo forecast horizon.

Cả hai baseline đều hỗ trợ cùng giao diện dự báo với các mô hình PatchTST, phục vụ quy trình training và evaluation thống nhất.

In [6]:
# 1. LSTM Baseline
class SimpleLSTMForecaster(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        num_layers,
        horizon
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_dim,
            horizon * input_dim
        )

    def forward(self, x, return_latents=False):

        _, (h_n, _) = self.lstm(x)

        z = h_n[-1]

        out = self.fc(z)

        if return_latents:
            return out, z, z

        return out


# 2. DLinear Baseline — Author's implementation
class DLinearForecaster(nn.Module):

    def __init__(
        self,
        input_dim,
        seq_len,
        horizon
    ):
        super().__init__()

        self.linears = nn.ModuleList([
            nn.Conv1d(
                1, 1,
                kernel_size=seq_len,
                bias=True
            )
            for _ in range(input_dim)
        ])

        self.horizon = horizon
        self.seq_len = seq_len
        self.input_dim = input_dim

    def forward(self, x, return_latents=False):

        outs = []

        for i, linear in enumerate(self.linears):

            xi = x[:, :, i].unsqueeze(1)

            if xi.shape[-1] < self.seq_len:
                pad = self.seq_len - xi.shape[-1]
                xi = torch.nn.functional.pad(
                    xi, (pad, 0)
                )

            yi = linear(xi)

            if yi.shape[-1] == 1 and self.horizon > 1:
                yi = yi.expand(
                    -1, -1, self.horizon
                )

            outs.append(yi)

        out = torch.cat(outs, dim=1)

        out = out.permute(
            0, 2, 1
        ).reshape(x.size(0), -1)

        if return_latents:
            return out, out, out

        return out

## 7. Unified Ablation Model Factory

Hàm `create_ablation_model()` cung cấp giao diện thống nhất để khởi tạo sáu mô hình trong Ablation Study.

Hàm nhận vào tên mô hình, số chiều đầu vào, patch length và forecast horizon.

Đầu ra bao gồm:

- `model`: Mô hình PyTorch được khởi tạo.
- `koop_attr`: Tên thuộc tính chứa toán tử Koopman nếu có.
- `lyap_weight`: Hệ số chính quy hóa Lyapunov.

Các mô hình không áp dụng chính quy hóa Lyapunov sử dụng hệ số bằng 0.

`KoopPatchTST_NoLyap` vẫn sử dụng toán tử Koopman ổn định nhưng không có thành phần phạt Lyapunov trong loss.

`KoopPatchTST_Full` sử dụng cùng kiến trúc và hệ số chính quy hóa Lyapunov bằng 0.1.

Model Factory được sử dụng bởi quy trình huấn luyện Ablation để bảo đảm cách khởi tạo nhất quán giữa các cấu hình thực nghiệm.

In [7]:
# 1. Unified Model Factory
def create_ablation_model(
    model_name: str,
    input_dim: int,
    patch_len: int,
    horizon: int
):
    """
    Create a model according to the author's
    Ablation Study configuration.

    Returns:
        model: PyTorch model
        koop_attr: Koopman attribute or None
        lyap_weight: Lyapunov regularization weight
    """

    if model_name not in MODEL_VARIANTS:
        raise ValueError(
            f"Unknown model: {model_name}"
        )

    if min(input_dim, patch_len, horizon) <= 0:
        raise ValueError(
            "input_dim, patch_len and horizon must be positive."
        )

    koop_attr = None
    lyap_weight = 0.0

    # 1. DLinear
    if model_name == "DLinear":

        model = DLinearForecaster(
            input_dim=input_dim,
            seq_len=patch_len,
            horizon=horizon
        )

    # 2. LSTM
    elif model_name == "LSTM":

        model = SimpleLSTMForecaster(
            input_dim=input_dim,
            hidden_dim=LSTM_HIDDEN_DIM,
            num_layers=LSTM_NUM_LAYERS,
            horizon=horizon
        )

    # 3. PatchTST Baseline
    elif model_name == "PatchTST":

        model = PatchTSTBaseline(
            input_dim=input_dim,
            seq_len=patch_len,
            horizon=horizon,
            patch_len=patch_len,
            d_model=D_MODEL
        )

    # 4. PatchTST + Unconstrained Linear Latent
    elif model_name == "PatchTST+LinearLatent":

        model = PatchTST_UnconstrainedLatent(
            input_dim=input_dim,
            seq_len=patch_len,
            horizon=horizon,
            patch_len=patch_len,
            d_model=D_MODEL
        )

    # 5. Koopman without Lyapunov regularization
    elif model_name == "KoopPatchTST_NoLyap":

        model = Koopformer_PatchTST(
            input_dim=input_dim,
            seq_len=patch_len,
            horizon=horizon,
            patch_len=patch_len,
            d_model=D_MODEL
        )

        koop_attr = "koop"
        lyap_weight = 0.0

    # 6. Full DeepKoopFormer
    elif model_name == "KoopPatchTST_Full":

        model = Koopformer_PatchTST(
            input_dim=input_dim,
            seq_len=patch_len,
            horizon=horizon,
            patch_len=patch_len,
            d_model=D_MODEL
        )

        koop_attr = "koop"
        lyap_weight = LYAP_WEIGHT

    return model, koop_attr, lyap_weight